# SMS SPAM 

In [9]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import string
import nltk
from sklearn.model_selection import train_test_split
nltk.download('stopwords', 'punkt')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to punkt...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
def clean_data(corpus_ensemble_doc):
    for i in range(len(corpus_ensemble_doc)):
       corpus_ensemble_doc[i] = corpus_ensemble_doc[i].lower()
    for i in range(len(corpus_ensemble_doc)):
        for c in string.punctuation:
            x = corpus_ensemble_doc[i].replace(c, " ")
            corpus_ensemble_doc[i]=x
    stopwords_anglais = stopwords.words('english')
    for i in range(len(corpus_ensemble_doc)):
        L = corpus_ensemble_doc[i].split()
        for word in L:
            if word in stopwords_anglais:
                while word in L:
                    L.remove(word)
        corpus_ensemble_doc[i] = " ".join(L)
    return corpus_ensemble_doc

In [11]:
def tf(terme, corpus, numero_document):
    x=corpus[numero_document].count(terme)
    y=len(corpus[numero_document].split())
    return x/y

def idf(terme, corpus, numero_document):
    D=len(corpus)
    d=0
    for document in corpus:
        if terme in document:
            d+=1
    TF_value=tf(terme, corpus, numero_document)
    return TF_value * np.log(1+ (D/d)) 

def cles_correspondante_a_valeur(valeur, dictionnaire):
    for cle in dictionnaire.keys():
        if dictionnaire[cle] == valeur:
            return cle

In [12]:
def matrice_sparse(dictionnaire, corpus_ensemble_documents):
    M = np.zeros((len(corpus_ensemble_documents), len(dictionnaire.values())))
    for i in range(len(corpus_ensemble_documents)):
        for j in dictionnaire.values():
            x=cles_correspondante_a_valeur(j, dictionnaire)
            M[i][j] = idf(x, corpus_ensemble_documents, i)
    return M

In [13]:
def affiche(M):
    (n,p) = M.shape
    for i in range(n):
        for j in range(p):
            M[i,j] = round(M[i,j], 2)
    print(M)

### Application de la régression Logistique sur les spams

In [15]:
spams = pd.read_table("data/SMSSpamCollection.txt", sep="\t",  header=0)
spamsTrain, spamsTest = train_test_split(spams, train_size=0.7, random_state=1)
parseur = CountVectorizer()
XTrain = parseur.fit_transform(spamsTrain["message"])
mdtTrain = XTrain.toarray()
modelFirst = LogisticRegression()
modelFirst.fit(mdtTrain, spamsTrain["classe"])
score1=modelFirst.score(mdtTrain, spamsTrain["classe"])
print("Score du modèle de base : ", score1)
mdtTest = parseur.transform(spamsTest["message"])
score2=modelFirst.score(mdtTest, spamsTest["classe"])
print("Score du modèle de base sur le test : ", score2)

Score du modèle de base :  0.997948717948718
Score du modèle de base sur le test :  0.9814593301435407


In [ ]:
spamsTrain, spamsTest = train_test_split(spams, train_size=0.7, random_state=1)
parseur = CountVectorizer(binary=True)
XTrain = parseur.fit_transform(spamsTrain["message"])
mdtTrain = XTrain.toarray()
modelFirst = LogisticRegression()
modelFirst.fit(mdtTrain, spamsTrain["classe"])
score3=modelFirst.score(mdtTrain, spamsTrain["classe"])
print("Score du modèle de base : ", score1)
mdtTest = parseur.transform(spamsTest["message"])
score4=modelFirst.score(mdtTest, spamsTest["classe"])
print("Score du modèle de base sur le test : ", score2)

Score du modèle de base :  0.9971794871794872
Score du modèle de base sur le test :  0.9844497607655502


In [17]:
from sklearn import metrics
parseurBis = CountVectorizer(binary=True, stop_words='english')
XTrainBis = parseurBis.fit_transform(spamsTrain["message"])
mdtTrainBis = XTrainBis.toarray()
modelBis = LogisticRegression()
modelBis.fit(mdtTrainBis, spamsTrain["classe"])
mdtTestBis = parseurBis.transform(spamsTest["message"])
score5=modelBis.score(mdtTrainBis, spamsTrain["classe"])
print("Score du modèle avec suppression des stop words : ", score5)
score6 = modelBis.score(mdtTestBis, spamsTest["classe"])
print("Score du modèle avec suppression des stop words sur le test : ", score6)

Score du modèle avec suppression des stop words :  0.9958974358974358
Score du modèle avec suppression des stop words sur le test :  0.9748803827751196


In [18]:
parseur3 = TfidfVectorizer(stop_words='english')
XTrain3 = parseur3.fit_transform(spamsTrain["message"])
mdtTrain3 = XTrain3.toarray()
model3 = LogisticRegression()
model3.fit(mdtTrain3, spamsTrain["classe"])
mdtTest3 = parseur3.transform(spamsTest["message"])
score7=model3.score(mdtTrain3, spamsTrain["classe"])
print("Score du modèle avec tf-idf : ", score7)
score8=model3.score(mdtTest3, spamsTest["classe"])
print("Score du modèle avec tf-idf sur le test : ", score8)

Score du modèle avec tf-idf :  0.9656410256410256
Score du modèle avec tf-idf sur le test :  0.9623205741626795
